## Tahap 0.1 — Lingkungan

**Dasar naskah:** Bab III (lingkungan komputasi: Google Colab) dan Bab III tahap 10 (GPU **T4**). `REVISI_30OKT.md` R1 (data CSV), R2 (folder kerja `MyDrive/skripsi_wondr_v4/`; `skripsi_wondr_v3/` tidak disentuh), R11 (seluruh pelatihan di T4). `CLAUDE.md` Bagian 3.1, 3.3, 3.4.

**Package:** `os`, `sys`, `subprocess`, `importlib`, `warnings`, `pathlib`, `json`, `time`, `platform` (bawaan Python); `numpy`, `pandas`, `torch`, `transformers`, `matplotlib` (di-*import*); `scikit-learn`, `scipy`, `openpyxl`, `pyarrow` (dipasang bila belum ada, versinya dicatat); `google.colab.drive` (*mount* Google Drive).

Jalankan sel 1/2 lalu sel 2/2 **paling awal** setiap kali *runtime* dimulai atau di-*restart*, sebelum Tahap 0.2.

In [ ]:
# ══ Tahap 0.1 — Lingkungan (sel 1/2: variabel lingkungan, folder kerja, pemasangan package) ══
# Package: os, sys, subprocess, importlib, warnings, pathlib (bawaan Python); google.colab.drive
import os, sys, subprocess, importlib.util, warnings
from pathlib import Path
from google.colab import drive

# 1) Variabel lingkungan, diatur SEBELUM torch/transformers di-import (3.3–3.4)
_cuda_dulu = "torch" in sys.modules and sys.modules["torch"].cuda.is_initialized()
_cublas_dulu = os.environ.get("CUBLAS_WORKSPACE_CONFIG")
os.environ.update({
    "CUBLAS_WORKSPACE_CONFIG": ":4096:8",                                          # determinisme cuBLAS
    "TOKENIZERS_PARALLELISM": "false",                                             # senyap: tokenizer
    "TRANSFORMERS_VERBOSITY": "error", "TRANSFORMERS_NO_ADVISORY_WARNINGS": "1",   # senyap: transformers
    "HF_HUB_DISABLE_PROGRESS_BARS": "1", "HF_DATASETS_DISABLE_PROGRESS_BARS": "1", # senyap: HF Hub, datasets
    "TQDM_DISABLE": "1",                                                           # senyap: semua tqdm
    "USE_TF": "0", "TF_CPP_MIN_LOG_LEVEL": "3",                                    # jangan muat TensorFlow
})
warnings.filterwarnings("ignore")

# 2) Google Drive dan struktur folder (3.1); folder versi lama tidak disentuh
if not os.path.isdir("/content/drive/MyDrive"):
    try:
        drive.mount("/content/drive")
    except ValueError as e:
        raise RuntimeError("Mount gagal: pilih akun yang menyimpan folder skripsi, centang SEMUA izin, ulangi sel.") from e
AKAR = Path("/content/drive/MyDrive/skripsi_wondr_v4")    # R2: folder v4; skripsi_wondr_v3 tidak disentuh
AKAR.mkdir(exist_ok=True)                         # gagal jika MyDrive belum ter-mount
for sub in ["data", "lib", "hasil", "anotasi", "bab4/tabel", "bab4/gambar", "log"]:
    (AKAR / sub).mkdir(parents=True, exist_ok=True)
(AKAR / "lib" / "__init__.py").touch()            # agar "from lib.wondr_teks import ..." jalan
os.chdir(AKAR)                                     # agar %%writefile cukup memakai lib/...
if str(AKAR) not in sys.path:
    sys.path.insert(0, str(AKAR))

# 3) Pasang hanya package yang belum ada (di-import di sel 2/2)
PAKET = {"numpy": "numpy", "pandas": "pandas", "torch": "torch", "transformers": "transformers",
         "sklearn": "scikit-learn", "scipy": "scipy", "matplotlib": "matplotlib",
         "openpyxl": "openpyxl", "pyarrow": "pyarrow"}            # nama modul -> nama pip
_dipasang = [pip for mod, pip in PAKET.items() if importlib.util.find_spec(mod) is None]
if _dipasang:
    _r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_dipasang], capture_output=True, text=True)
    if _r.returncode != 0:
        raise RuntimeError("pip install gagal:\n" + _r.stderr[-2000:])

In [ ]:
# ══ Tahap 0.1 — Lingkungan (sel 2/2: import, GPU, dan log versi) ══
# Package: importlib, json, time, platform (bawaan Python); numpy, pandas, torch, transformers,
#          matplotlib (di-import); scikit-learn, scipy, openpyxl, pyarrow (versinya dicatat)
import importlib.metadata, json, time, platform
t0 = time.perf_counter()
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, transformers
from huggingface_hub.utils import logging as hf_logging

# 1) Matikan kebisingan yang tersisa setelah import
transformers.logging.set_verbosity_error()        # sembunyikan "Some weights ... newly initialized"
transformers.logging.disable_progress_bar()       # progress bar unduhan model
hf_logging.set_verbosity_error()                  # sembunyikan peringatan HF Hub "unauthenticated requests"
plt.rcParams.update({"savefig.dpi": 300, "savefig.bbox": "tight"})   # PNG 300 dpi (3.1)

# 2) GPU dan versi -> log/versi_package.json (acuan = sesi T4 pertama)
GPU_PENELITIAN = "T4"                              # R11: Bab III tahap 10; seluruh pelatihan di T4
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "tidak ada"
mem = torch.cuda.get_device_properties(0).total_memory / 1024**3 if gpu != "tidak ada" else 0.0
sesi = {"waktu": time.strftime("%Y-%m-%d %H:%M:%S"), "python": platform.python_version(), "gpu": gpu,
        "gpu_memori_gb": round(mem, 2), "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
        "package": {p: importlib.metadata.version(p) for p in [*PAKET.values(), "tokenizers", "huggingface-hub"]}}
log = AKAR / "log" / "versi_package.json"
lama = json.loads(log.read_text()) if log.exists() else {}
pertama = lama["pertama"] if GPU_PENELITIAN in lama.get("pertama", {}).get("gpu", "") else sesi
log.write_text(json.dumps({"pertama": pertama, "terbaru": sesi}, indent=2))
beda = [k for k in ("torch", "transformers", "tokenizers") if pertama["package"][k] != sesi["package"][k]]
beda += ["GPU"] if pertama["gpu"] != gpu else []

# 3) Ringkasan + cek (judul() dan cek() baru dibuat di Tahap 0.2)
csv, koma = AKAR / "data" / "review_wondr_bni_databaruu.csv", lambda x: f"{x:.1f}".replace(".", ",")   # R1
cublas_ok = not _cuda_dulu or _cublas_dulu == ":4096:8"
print(f"══════════ TAHAP 0.1 — LINGKUNGAN ══════════\nMasukan : data/{csv.name}\nKeluaran: log/{log.name}")
print(f"{'─' * 48}\nFolder kerja : {AKAR}\nPython       : {sesi['python']}\n"
      f"GPU          : {gpu} ({koma(mem)} GB) · CUDA {sesi['cuda']} · cuDNN {sesi['cudnn']}\n\n{'Package':<17}Versi")
print("\n".join(f"{k:<17}{v}{'  (baru dipasang)' * (k in _dipasang)}" for k, v in sesi["package"].items()))
print(f"{'─' * 48}\nCek otomatis:")
for nama, ok, aktual, target in [
        ("berkas data ada", csv.exists(), "ya" if csv.exists() else "tidak", "ya"),
        ("GPU", GPU_PENELITIAN in gpu, gpu, f"memuat {GPU_PENELITIAN}"),
        ("CUBLAS_WORKSPACE_CONFIG diatur sebelum CUDA aktif", cublas_ok, "ya" if cublas_ok else "tidak, restart runtime", "ya"),
        ("versi torch/transformers/tokenizers/GPU", not beda, "berubah: " + ", ".join(beda) if beda else "sama", "sama")]:
    print(f"  {'✅' if ok else '⚠️'} {nama} = {aktual} (target {target})")
print(f"Waktu: {koma(time.perf_counter() - t0)} detik")

## Tahap 0.2 — Konfigurasi dan fungsi bantu

**Dasar naskah:** Bab III (seluruh tahap; Tabel 3.1 dan 3.2). `CLAUDE.md` Bagian 4 dan keputusan KT-2, KT-3, KT-4; `REVISI_30OKT.md` R3 (*seed*: RS 1–10, PSO 11–20, model akhir 44–48), R4 (10 *run* per metode), R10 (`kembaran_dekat` dihapus), R11 (GPU T4), Bagian 3 (target data baru), Bagian 6 (nomor persamaan); `REVISI_BRODY.md` R15 (`brody_ambang` = 10) dan Bagian 5 (target Tahap 4 dan seterusnya).

**Package:** `importlib` (bawaan Python); `lib.bantu` (`json`, `math`, `os`, `random`, `time`, `pathlib`, `numpy`; `torch` dan `transformers` di `atur_seed()`).

Sel 1/2 menulis `lib/bantu.py` (tidak berubah dari v3). Sel 2/2 membuat `PATH`, `CFG`, dan `TARGET`; jalankan setelah Tahap 0.1 setiap kali *runtime* dimulai.

In [ ]:
%%writefile lib/bantu.py
# lib/bantu.py -- fungsi bantu untuk semua subtahap (Tahap 0.2; KT-4 disetujui 21-10-2026).
# Isinya fungsi saja; PATH, CFG, dan TARGET dibuat di sel 2/2 Tahap 0.2.
import json, math, os, random, time
from pathlib import Path
import numpy as np

__all__ = ["fmt_id", "rel", "mulai", "judul", "garis", "cek", "selesai",
           "simpan_tabel", "simpan_gambar", "simpan_json", "baca_json", "atur_seed"]
_AKAR = Path(__file__).resolve().parents[1]          # folder kerja .../skripsi_wondr_v3
_STATUS = {"t0": None, "cek_mulai": False}
_LEBAR = 48                                           # lebar garis pemisah blok ringkasan


def _angka(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def fmt_id(x, d=None):
    """Format angka Indonesia: 7204 -> '7.204'; 0.836 -> '0,8360'. Selain angka -> str(x)."""
    if not _angka(x):
        return str(x)
    if d is None:
        d = 0 if isinstance(x, (int, np.integer)) else 4
    return f"{x:,.{d}f}".replace(",", "_").replace(".", ",").replace("_", ".")


def _tampil(v, d=None):
    """Nilai untuk ringkasan: bilangan bulat tanpa desimal, pecahan d desimal, daftar dipisah ' · '."""
    if isinstance(v, np.ndarray):
        v = v.tolist()
    if isinstance(v, dict):
        return ", ".join(f"{k}: {_tampil(x, d)}" for k, x in v.items())
    if isinstance(v, (list, tuple)):
        return " · ".join(f"[{_tampil(x, d)}]" if isinstance(x, (list, tuple)) else _tampil(x, d) for x in v)
    return fmt_id(v) if isinstance(v, (int, np.integer)) else fmt_id(v, d)


def rel(p):
    """Path relatif terhadap folder kerja, untuk ditampilkan."""
    try:
        return str(Path(p).resolve().relative_to(_AKAR))
    except ValueError:
        return str(p)


def garis():
    print("─" * _LEBAR)


def mulai():
    """Dipanggil di awal sel: mulai menghitung waktu."""
    _STATUS["t0"] = time.perf_counter()


def judul(no, nama, masukan=None, keluaran=None):
    """Kepala blok ringkasan; masukan/keluaran berupa str atau list of str."""
    _STATUS["cek_mulai"] = False
    print(f"{'═' * 10} TAHAP {no} — {nama.upper()} {'═' * 10}")
    for label, isi in (("Masukan ", masukan), ("Keluaran", keluaran)):
        if isi:
            isi = [isi] if isinstance(isi, str) else list(isi)
            print(f"{label}: " + f"\n{' ' * 10}".join(isi))
    garis()


def _sama(a, t, tol):
    a = a.tolist() if isinstance(a, np.ndarray) else a
    t = t.tolist() if isinstance(t, np.ndarray) else t
    if _angka(a) and _angka(t):
        return abs(float(a) - float(t)) <= tol
    if isinstance(a, dict) and isinstance(t, dict):
        return a.keys() == t.keys() and all(_sama(a[k], t[k], tol) for k in t)
    if isinstance(a, (list, tuple)) and isinstance(t, (list, tuple)):
        return len(a) == len(t) and all(_sama(x, y, tol) for x, y in zip(a, t))
    return bool(a == t)


def cek(nama, aktual, target, toleransi=0, cara="=", d=None):
    """Cetak ✅/⚠️ beserta nilai aktual dan target, tanpa assert. cara: '=', '>', '>=', '<', '<='."""
    if not _STATUS["cek_mulai"]:
        garis()
        print("Cek otomatis:")
        _STATUS["cek_mulai"] = True
    if d is None and toleransi > 0:
        d = max(0, round(-math.log10(2 * toleransi)))  # 0,05 -> 1 desimal; 5e-5 -> 4 desimal
    if cara == "=":
        ok = _sama(aktual, target, toleransi)
    else:
        a, t = float(aktual), float(target)
        ok = {">": a > t, ">=": a >= t, "<": a < t, "<=": a <= t}[cara]
    tanda = "" if cara == "=" else cara + " "
    print(f"  {'✅' if ok else '⚠️'} {nama} = {_tampil(aktual, d)} (target {tanda}{_tampil(target, d)})")
    return bool(ok)


def selesai():
    """Baris terakhir blok ringkasan: lama sel berjalan sejak mulai()."""
    dt = time.perf_counter() - _STATUS["t0"] if _STATUS["t0"] else 0.0
    teks = (f"{fmt_id(dt, 1)} detik" if dt < 120 else
            f"{fmt_id(dt / 60, 1)} menit" if dt < 7200 else f"{fmt_id(dt / 3600, 2)} jam")
    print(f"Waktu: {teks}")


def simpan_tabel(df, nama, index=False):
    """CSV untuk Bab IV (angka mentah) -> bab4/tabel/<nama>.csv; mengembalikan path-nya."""
    p = _AKAR / "bab4" / "tabel" / f"{nama}.csv"
    df.to_csv(p, index=index)
    return p


def simpan_gambar(fig, nama):
    """PNG 300 dpi -> bab4/gambar/<nama>.png, lalu ditampilkan satu kali."""
    import matplotlib.pyplot as plt
    p = _AKAR / "bab4" / "gambar" / f"{nama}.png"
    fig.savefig(p, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return p


def _ke_json(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, (np.ndarray, Path)):
        return o.tolist() if isinstance(o, np.ndarray) else str(o)
    raise TypeError(f"tidak bisa disimpan ke JSON: {type(o).__name__}")


def simpan_json(obj, path):
    """Tulis JSON secara atomik (berkas .tmp lalu diganti), aman bila Colab terputus."""
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=_ke_json), encoding="utf-8")
    os.replace(tmp, path)
    return path


def baca_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def atur_seed(s):
    """Satu seed untuk semua sumber keacakan pelatihan (3.4), ditambah KT-2 dan KT-3 (21-10-2026)."""
    import torch, transformers
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    transformers.set_seed(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)         # KT-2: operasi GPU tak-deterministik -> error
    torch.backends.cuda.matmul.allow_tf32 = False    # KT-3: FP32 penuh; API lama ikut mengatur API baru,
    torch.backends.cudnn.allow_tf32 = False          #       API baru saja tidak mematikan TF32 konv/RNN cuDNN


def uji_bantu(s):
    """Uji Tahap 0.2: atur_seed(s) dua kali -> angka acak sama; status mode deterministik dan TF32."""
    import torch
    atur_seed(s); a = np.random.rand(3), torch.rand(3)
    atur_seed(s); b = np.random.rand(3), torch.rand(3)
    return {"ulang_sama": all(np.array_equal(x, y) for x, y in zip(a, b)),
            "deterministik": torch.are_deterministic_algorithms_enabled(),
            "tf32": [torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32]}

In [ ]:
# ══ Tahap 0.2 — Konfigurasi dan fungsi bantu (sel 2/2: PATH, CFG, TARGET) ══
# Package: importlib (bawaan Python); lib.bantu (fungsi bantu dari sel 1/2)
import importlib, lib.bantu
importlib.reload(lib.bantu)                         # pakai isi lib/bantu.py yang terbaru
from lib.bantu import *
mulai()

# 1) Lokasi folder dan berkas (3.1)
PATH = {"akar": AKAR, **{k: AKAR / k for k in ("data", "lib", "hasil", "anotasi", "log")},
        "tabel": AKAR / "bab4/tabel", "gambar": AKAR / "bab4/gambar", "csv": AKAR / "data/review_wondr_bni_databaruu.csv"}

# 2) Konfigurasi tetap (CLAUDE.md Bagian 4; REVISI_30OKT R3, R4, R11; REVISI_BRODY R15) -- jangan diubah
CFG = {
    "gpu": "T4",                                                    # R11; Bab III tahap 10
    "checkpoint": "indobenchmark/indobert-base-p1",                 # Bab III tahap 7; Wilie dkk. (2020)
    "tau": 4, "kelas": {0: "Non-Positif", 1: "Positif"},            # pers. (2.1)-(2.2): rating >= 4 -> 1
    "brody_ambang": 10,                                             # Brody & Diakopoulos (2011); Bab II 2.2.4
    "proporsi": {"latih": 0.70, "validasi": 0.15, "uji": 0.15},     # Bab III tahap 6
    "seed_bagi": 42, "kelompok_panjang": ("1-2", "3-10", ">10"),    # Bab III tahap 6
    "q": 100, "b": 32,                                              # pers. (2.9): L_max dari data latih saja
    "cl_lipatan": 5, "cl_seed_lipatan": 42, "cl_seed_latih": 42,    # Bab III tahap 8; Keputusan teknis
    "seed_anotasi": 2026,                                           # Keputusan: urutan lembar anotasi (tahap 9)
    "B": 16, "E": 4, "p": 2, "v": 1.0,                              # Tabel 3.1: batch, epoch, patience, norma
    "adam_betas": (0.9, 0.999), "adam_eps": 1e-8,                   # pers. (2.18); bawaan PyTorch
    "default": {"lr": 2e-5, "wr": 0.1, "wd": 0.01},                 # Tabel 3.1; Devlin dkk.; Sun dkk.
    "ruang": {"lr": (1e-5, 5e-5), "wr": (0.0, 0.2), "wd": (0.0, 0.3)},   # Tabel 3.1; pers. (2.29); lr skala log
    "seed_fitness": (42, 43),                                       # Bab III tahap 10; pers. (2.23), (2.25)-(2.26)
    "rs_M": 50, "rs_seeds": tuple(range(1, 11)),                    # Bab III tahap 11; R3-R4: 10 run, seed 1-10
    "pso": {"s": 10, "G_max": 4, "omega": 0.729, "c1": 1.49445, "c2": 1.49445, "V_max": 1.0},   # Tabel 3.2
    "pso_seeds": tuple(range(11, 21)),                              # Bab III tahap 12; R3-R4: 10 run, seed 11-20
    "seed_akhir": (44, 45, 46, 47, 48),                             # Bab III tahap 13; R3, R6
    "R": 10_000, "seed_bootstrap": 42,                              # Bab III tahap 15; pers. (2.43)-(2.45)
    "alpha": 0.05, "m_uji": 3,                                      # pers. (2.48)-(2.49): alpha' = 0,05/3
}

# 3) Angka target (REVISI_30OKT Bagian 3, diganti sebagian oleh REVISI_BRODY Bagian 5; nomor tahap mengikuti Bab III)
TARGET = {
    "1.1": {"baris": 9672, "kolom": ["nama", "rating", "ulasan", "tanggal", "likes", "balasan_dev", "versi"],
            "tanggal": ("2026-07-19", "2026-10-04"), "versi": ["1.6.0"],
            "per_bulan": {"2026-07": 1734, "2026-08": 3857, "2026-09": 3525, "2026-10": 556}},
    "1.2": {"kosong": {"balasan_dev": 4474}, "dibalas_pct": {"1-4": 99.6, "5": 30.8}, "likes_nol_pct": 95.1,
            "spearman": -0.2404, "r3_panjang": (315, 479), "median_kata": [15, 15, 11, 2, 2], "emoji_simbol": 549,
            "huruf_ulang": (326, 3.4), "baca_ulang": 797, "duplikat_pct": 36.7, "label_ganda": (42, 2721),
            "nama_unik": 9573},
    "2": {"baris": 9672, "kolom": ["review_id", "ulasan", "rating", "tanggal", "versi"]},
    "3": {"per_rating": {1: 1713, 2: 372, 3: 479, 4: 659, 5: 6449}, "positif": (7108, 73.49),
          "non_positif": (2564, 26.51)},
    "4": {"contoh_bab2": ["bagus bgt! tp kadang2 error, nunggu lama", "mantap"],
          "berubah": [66, 549, 2001, 326, 797, 547], "kosong": 92, "kosong_rating": {5: 85, 4: 6, 1: 1}, "hasil": 9580,
          "kamus": {"kelompok_memanjang": 176, "buang_1_huruf": 0, "buang_frekuensi": 88,
                    "buang_kanonik_memanjang": 0, "dipakai": 88},
          "memanjang": {"total": 381, "kamus": 266, "cadangan": 115, "bentuk": (260, 158, 102)}},   # R13-R15
    "5": {"kunci": 5684, "kunci_seri": 11, "ulasan_seri": 24, "dokumen": 5673, "dibuang": (3907, 3784, 99, 24),
          "positif_pct": (73.2, 57.0), "gabungan": (49, 0)},        # gabungan: kunci terdampak, memuat angka (R9)
    "6.1": {"grup": 5563, "latih": (3969, 1708, 2261), "validasi": (852, 366, 486), "uji": (852, 367, 485)},
    "6.2": {"bocor": 0, "non_positif_pct": (43.0, 43.0, 43.1), "p_label": 0.9987,
            "p_panjang": 0.9561, "panjang_pct": {"latih": (13.5, 47.7, 38.8), "validasi": (14.4, 46.8, 38.7),
                                                 "uji": (13.3, 48.0, 38.7)},
            "dampak": {"ulasan_latih": 6798, "kamus_latih": {"kelompok_memanjang": 131, "buang_1_huruf": 0,
                       "buang_frekuensi": 64, "buang_kanonik_memanjang": 0, "dipakai": 67},
                       "berbeda": {"latih": 4, "validasi": 9, "uji": 4},
                       "berbeda_id": {"validasi": [1868, 2732, 4911, 6773, 7582, 7593, 7609, 8671, 9371],
                                      "uji": [3668, 5741, 7605, 7928]}}},   # R16: cek dampak kamus Brody
    "7": {"L_max": 128, "terpotong": {"latih": 0, "validasi": None, "uji": None},
          "terpotong_id": []},                                      # validasi/uji diisi setelah output 7 disetujui
    "8.1": {"T": 996, "T_w": 99},                                   # n = 3.969, rho = 0,1
    "8.3": {"t0": 0.6167, "t1": 0.6667, "C": [[2, 1], [1, 2]], "kandidat": [3, 6]},    # contoh Tabel 2.9
    "9.2": {"kappa_contoh": 0.70, "toleransi": 1e-12},              # contoh Bab II 2.3.5
    "10": {"T": 996, "T_w": 99}, "11": {"run": 10, "kandidat": 50, "pelatihan": 100},   # 11: kandidat/pelatihan per run
    "12.1": {"iter1": (5.6237, 3.6355), "iter2": (2.6718, 3.3698), "gbest_iter2": 9.8923,
             "toleransi": 5e-4, "run": 10, "evaluasi": 50},         # contoh Tabel 2.17; evaluasi per run
    "13": {"pelatihan": 15, "toleransi": 1e-4}, "14.1": {"n_uji": 852},
    "15": {"f1_contoh": 0.6703, "persentil": (0.8333, 99.1667), "toleransi": 1e-12},   # contoh Tabel 2.19
}

# 4) Uji singkat fungsi bantu, lalu ringkasan
u = lib.bantu.uji_bantu(CFG["seed_fitness"][0])
rentang = lambda s: f"{s[0]}–{s[-1]}" if list(s) == list(range(s[0], s[-1] + 1)) else ", ".join(map(str, s))
judul("0.2", "Konfigurasi dan fungsi bantu", masukan="lib/bantu.py", keluaran="PATH, CFG, TARGET (di memori)")
print(f"CFG {len(CFG)} entri · TARGET {len(TARGET)} subtahap · PATH {len(PATH)} lokasi · "
      f"fungsi bantu {len(lib.bantu.__all__)}")
print(f"seed: fitness {rentang(CFG['seed_fitness'])} · RS {rentang(CFG['rs_seeds'])} · PSO {rentang(CFG['pso_seeds'])} · "
      f"model akhir {rentang(CFG['seed_akhir'])} · GPU {CFG['gpu']}")
cek("fmt_id(7204) · fmt_id(0.836)", [fmt_id(7204), fmt_id(0.836)], ["7.204", "0,8360"])
cek("atur_seed() dua kali, angka acak sama", "ya" if u["ulang_sama"] else "tidak", "ya")
cek("mode deterministik PyTorch (KT-2)", "aktif" if u["deterministik"] else "mati", "aktif")
cek("TF32 matmul · cuDNN (KT-3)", ["nyala" if x else "mati" for x in u["tf32"]], ["mati", "mati"])
S3 = [set(CFG[k]) for k in ("seed_fitness", "rs_seeds", "pso_seeds", "seed_akhir")]
cek("run RS · PSO · seed akhir (R3) · tidak bertumpuk · kunci lama terhapus",
    [len(CFG["rs_seeds"]), len(CFG["pso_seeds"]), len(CFG["seed_akhir"]),
     "ya" if sum(map(len, S3)) == len(set().union(*S3)) else "tidak",
     "ya" if not ({"rs_seed", "kembaran_dekat"} & set(CFG) or "seed" in CFG["pso"]) else "tidak"],
    [TARGET["11"]["run"], TARGET["12.1"]["run"], TARGET["13"]["pelatihan"] // 3, "ya", "ya"])   # 15 pelatihan / 3 skema
selesai()

## Tahap 1.1 — Muat data mentah

**Dasar naskah:** Bab III tahap 1 (Pengumpulan Data) dan subbab 3.1: `review_wondr_bni_databaruu.csv`, 9.672 ulasan, 19 Juli–4 Oktober 2026, versi aplikasi 1.6.0, disimpan apa adanya dan tidak diubah (`REVISI_30OKT.md` R1). `review_id` dibuat di Tahap 2 sesuai Bab III tahap 2 (`CLAUDE.md` Bagian 10, keputusan no. 7).

**Package:** `pandas` (baca .csv, olah tabel), `pyarrow` (mesin simpan .parquet); `lib.bantu` (fungsi bantu Tahap 0.2).

In [ ]:
# ══ Tahap 1.1 — Muat data mentah ══
# Package: pandas (baca .csv, olah tabel), pyarrow (mesin simpan .parquet)
import pandas as pd
mulai()
T = TARGET["1.1"]
BULAN = ["Januari", "Februari", "Maret", "April", "Mei", "Juni", "Juli",
         "Agustus", "September", "Oktober", "November", "Desember"]
tgl_id = lambda t: f"{t.day} {BULAN[t.month - 1]} {t.year}"       # 2026-07-19 -> "19 Juli 2026"

# 1) Baca data mentah (.csv, R1); versi dibaca sebagai teks agar "1.6.0" tidak pernah ditafsirkan sebagai angka
mentah = pd.read_csv(PATH["csv"], dtype={"versi": str})

# 2) Buang kolom indeks bawaan (Unnamed*), bila ada; isi tidak diubah, review_id dibuat di Tahap 2
buang = [c for c in mentah.columns if str(c).startswith("Unnamed")]
df = mentah.drop(columns=buang)

# 3) Simpan, lalu pastikan berkas yang dibaca ulang identik
keluar = PATH["hasil"] / "tahap01_mentah.parquet"
df.to_parquet(keluar, index=False)
identik = df.equals(pd.read_parquet(keluar))

# 4) Ringkasan + cek
tgl = pd.to_datetime(df["tanggal"])                                # hanya untuk ringkasan; kolom asli tetap
per_bulan = {k: int(v) for k, v in tgl.dt.strftime("%Y-%m").value_counts().sort_index().items()}
versi = sorted(df["versi"].astype(str).unique())
judul("1.1", "Muat data mentah", masukan=f"{rel(PATH['csv'])} ({fmt_id(len(mentah))} baris, {mentah.shape[1]} kolom)",
      keluaran=f"{rel(keluar)} ({fmt_id(len(df))} baris, {df.shape[1]} kolom)")
print(f"Kolom dibuang  : {', '.join(map(str, buang)) or '-'}")
print(f"Rentang tanggal: {tgl_id(tgl.min())} – {tgl_id(tgl.max())}")
print(f"Versi aplikasi : {', '.join(versi)}\n\n{'Bulan':<16}{'Ulasan':>7}")
for k, v in per_bulan.items():
    print(f"{BULAN[int(k[5:]) - 1] + ' ' + k[:4]:<16}{fmt_id(v):>7}")
print(f"{'Total':<16}{fmt_id(sum(per_bulan.values())):>7}")
cek("jumlah baris", len(df), T["baris"])
cek("nama dan urutan kolom", "sesuai" if list(df.columns) == T["kolom"] else ", ".join(map(str, df.columns)), "sesuai")
cek("rentang tanggal", (str(tgl.min().date()), str(tgl.max().date())), T["tanggal"])
cek("nilai unik versi", versi, T["versi"])
cek("ulasan per bulan", per_bulan, T["per_bulan"])
cek("parquet identik dengan isi .csv", "ya" if identik else "tidak", "ya")
selesai()

## Tahap 1.2 — Tulis `lib/wondr_teks.py` dan audit kualitas data *(audit: program saja; data tidak diubah)*

**Dasar naskah:** Bab III tahap 1 (data mentah tidak diubah) dan tahap 2 (alasan membuang `nama`, `balasan_dev`, `likes`); pola teks sama dengan Tahap 4 (Bab II 2.2.2, 2.2.4, 2.2.5). Audit tidak dibahas di naskah. Sel 2/2 tidak berubah dari v3; hanya angka `TARGET["1.2"]` yang mengikuti data baru (`REVISI_30OKT.md` Bagian 3 dan 4: ✅🔢). Sel 1/2 mengikuti `REVISI_BRODY.md` Bagian 3 (R13–R15): `LANGKAH` dan `bersihkan(s)` lama diganti normalisasi pemanjangan kata (kamus Brody & Diakopoulos, 2011, lalu cadangan ≥ 3 → 2) dengan `langkah(kamus)`, `bersihkan(s, kamus)`, `kamus_brody()`, `pemanjangan()`, `ringkas()`, `sebelum_kamus()`; pola dan fungsi Tahap 5–6 tidak berubah.

**Package:** `json`, `importlib` (bawaan Python); `pandas` (olah tabel); `scipy.stats.spearmanr` (korelasi *likes*–*rating*); `lib.wondr_teks` (pola `EMO`, `SIMBOL`, `HURUF`, `BACA`; memakai `re`, `unicodedata`, `numpy`, `pandas`); `lib.bantu`.

Sel 1/2 menulis kode acuan Bagian 7 `CLAUDE.md` dan Bagian 3 `REVISI_BRODY.md` ke `lib/wondr_teks.py` (fungsi saja). Sel 2/2 mengaudit data tanpa mengubahnya.

In [ ]:
%%writefile lib/wondr_teks.py
# lib/wondr_teks.py -- kode acuan Bagian 7 CLAUDE.md dan Bagian 3 REVISI_BRODY.md;
# dipakai Tahap 1.2, 4, 5, 6, dan 6.2 (cek dampak kamus).
import re, unicodedata
from collections import Counter, defaultdict
import numpy as np, pandas as pd

# -- Pola teks (Tahap 1.2 dan Tahap 4 memakai pola yang sama) --
EMO = re.compile("[\U0001F000-\U0001FAFF\U0001F1E6-\U0001F1FF\u2600-\u27BF\u2B00-\u2BFF"
                 "\u2300-\u23FF\u2190-\u21FF\u2900-\u297F\u3030\u303D\u3297\u3299"
                 "\u00A9\u00AE\u203C\u2049\u2122\u2139\u24C2\u25AA-\u25FE\uFE0F\u200D"
                 "\U000E0020-\U000E007F]")
SIMBOL = re.compile("[\u2600-\u27BF\u2B00-\u2BFF\uFE0F\u200D\U0001F3FB-\U0001F3FF]")
HURUF = re.compile(r"([^\W\d_])\1{2,}")      # huruf yang berulang >= 3 kali (hanya huruf)
BACA = re.compile(r"[!?.,;:]{2,}")           # tanda baca berurutan >= 2

# -- Tahap 4 langkah 4: normalisasi pemanjangan kata (Brody & Diakopoulos, 2011), lalu cadangan >=3 -> 2 --
KATA = re.compile(r"[^\W\d_]+")    # kata = deretan huruf; angka dan tanda baca tidak termasuk kata

def ringkas(w):                    # semua huruf berurutan yang sama -> satu (hanya kunci kelompok)
    return re.sub(r"([^\W\d_])\1+", r"\1", w)

def kamus_brody(teks, ambang=10):
    """Kamus {kunci ringkas: bentuk kanonik} dari frekuensi kata seluruh teks; label tidak dipakai."""
    frek = Counter(w for t in teks for w in KATA.findall(t))
    grup = defaultdict(dict)
    for w, n in frek.items():
        grup[ringkas(w)][w] = n
    kamus, info = {}, {"kelompok_memanjang": 0, "buang_1_huruf": 0, "buang_frekuensi": 0, "buang_kanonik_memanjang": 0}
    for k in sorted(grup):
        g = grup[k]
        if not any(HURUF.search(w) for w in g):                 # kelompok tanpa bentuk memanjang
            continue
        info["kelompok_memanjang"] += 1
        kan, n = sorted(g.items(), key=lambda x: (-x[1], len(x[0]), x[0]))[0]   # tersering; seri: terpendek, abjad
        if len(kan) == 1:
            info["buang_1_huruf"] += 1
        elif n < ambang:
            info["buang_frekuensi"] += 1
        elif HURUF.search(kan):
            info["buang_kanonik_memanjang"] += 1
        else:
            kamus[k] = kan
    return kamus, info

def pemanjangan(s, kamus):
    """Kata berhuruf berulang >= 3: bentuk kanonik bila kelompoknya ada di kamus; selain itu >= 3 -> 2."""
    def ganti(m):
        w = m.group()
        return kamus.get(ringkas(w), HURUF.sub(r"\1\1", w)) if HURUF.search(w) else w
    return KATA.sub(ganti, s)

def langkah(kamus):  # (nama, fungsi) -- urutan Tahap 4, jangan diubah
    return [
        ("1. Normalisasi Unicode NFKC", lambda s: unicodedata.normalize("NFKC", s)),
        ("2. Hapus emoji dan simbol", lambda s: SIMBOL.sub(" ", EMO.sub(" ", s))),
        ("3. Case folding", lambda s: s.lower()),
        ("4. Pemanjangan kata (Brody, >=3->2)", lambda s: pemanjangan(s, kamus)),
        ("5. Tanda baca berulang -> 1", lambda s: BACA.sub(lambda m: m.group(0)[0], s)),
        ("6. Rapikan spasi", lambda s: re.sub(r"\s+", " ", s).strip()),
    ]

def sebelum_kamus(s):              # langkah 1-3 saja: teks yang dihitung untuk membentuk kamus
    for _, f in langkah({})[:3]:
        s = f(s)
    return s

def bersihkan(s, kamus):
    s = str(s)
    for _, f in langkah(kamus):
        s = f(s)
    return s

# -- Tahap 5: deduplikasi --
def kunci_dedup(s):
    s = re.sub(r"[^a-z0-9 ]", " ", str(s))   # sisakan a-z, 0-9, spasi
    s = re.sub(r"(.)\1+", r"\1", s)          # karakter berurutan yang sama -> satu
    return re.sub(r"\s+", " ", s).strip()

def deduplikasi(d):
    d = d.copy()
    d["dedup_key"] = d["text_clean"].map(kunci_dedup)
    d = d[d["dedup_key"] != ""]
    suara = (d.groupby(["dedup_key", "label"]).size().unstack(fill_value=0)
               .reindex(columns=[0, 1], fill_value=0))
    suara.columns = ["n_np", "n_p"]
    seri = suara["n_np"] == suara["n_p"]
    suara["mayor"] = (suara["n_p"] > suara["n_np"]).astype(int)
    sah = suara[~seri]
    x = d[d["dedup_key"].isin(sah.index)].copy()
    x = x[x["label"] == x["dedup_key"].map(sah["mayor"])]          # anggota berlabel mayoritas
    frek = x.groupby(["dedup_key", "text_clean"]).size().rename("_f").reset_index()
    x = x.merge(frek, on=["dedup_key", "text_clean"])
    wakil = (x.sort_values(["dedup_key", "_f", "review_id"], ascending=[True, False, True])
               .groupby("dedup_key").head(1)["review_id"])           # teks tersering, lalu id terkecil
    hasil = d[d["review_id"].isin(set(wakil))].copy()
    hasil["label"] = hasil["dedup_key"].map(sah["mayor"]).astype(int)
    return hasil.sort_values("review_id").reset_index(drop=True), suara, seri

# -- Tahap 6: pembagian data berbasis grup --
KATA_FUNGSI = {"dan", "yang", "nya", "ini", "itu", "ya", "sih", "deh", "juga", "aja", "saja", "nih",
               "kok", "dong", "lah", "di", "ke", "dari", "untuk", "pada", "dengan", "yg", "jg", "utk", "dgn"}

def kunci_grup(s):                      # hanya untuk pengelompokan; teks tidak diubah
    k = set(str(s).split())
    inti = k - KATA_FUNGSI
    return " ".join(sorted(inti)) if inti else " ".join(sorted(k))

def kelompok_panjang(n_kata):
    return "1-2" if n_kata <= 2 else ("3-10" if n_kata <= 10 else ">10")

PROPORSI = {"latih": 0.70, "validasi": 0.15, "uji": 0.15}   # urutan kunci menentukan pemecah seri

def bagi_data(dd, seed=42):
    dd = dd.copy()
    dd["group_key"] = dd["text_clean"].map(kunci_grup)
    dd["n_kata"] = dd["text_clean"].str.split().str.len()
    label_id_kecil = dd.sort_values("review_id").groupby("group_key")["label"].first()
    g = (dd.groupby("group_key")
           .agg(n=("review_id", "size"), n_p=("label", "sum"), kata=("n_kata", "median"))
           .reset_index())
    mayor = np.where(g.n_p * 2 > g.n, 1, np.where(g.n_p * 2 < g.n, 0, -1))
    g["label_grup"] = np.where(mayor == -1, g.group_key.map(label_id_kecil).to_numpy(), mayor).astype(int)
    g["kelompok_panjang"] = g["kata"].map(kelompok_panjang)
    rng = np.random.default_rng(seed)                     # satu generator untuk semua strata
    alokasi = {}
    for _, sub in g.groupby(["label_grup", "kelompok_panjang"], sort=True):
        sub = sub.iloc[rng.permutation(len(sub))]
        total = int(sub.n.sum())
        target = {b: p * total for b, p in PROPORSI.items()}
        terisi = {b: 0 for b in PROPORSI}
        for kunci, n in zip(sub.group_key, sub.n):
            b = min(terisi, key=lambda b: terisi[b] / target[b])   # keterisian terkecil
            alokasi[kunci] = b
            terisi[b] += int(n)
    dd["split"] = dd["group_key"].map(alokasi)
    dd["kelompok_panjang"] = dd["n_kata"].map(kelompok_panjang)
    return dd, g

In [ ]:
# ══ Tahap 1.2 — Audit kualitas data (sel 2/2; program saja, data tidak diubah) ══
# Package: json, importlib (bawaan Python); pandas; scipy.stats.spearmanr; lib.wondr_teks (pola teks)
import json, importlib
import pandas as pd
from scipy.stats import spearmanr
import lib.wondr_teks as wt
from lib.bantu import _tampil                              # format angka Indonesia untuk tabel
importlib.reload(wt)                                        # pakai isi lib/wondr_teks.py yang terbaru
mulai()
T = TARGET["1.2"]

# 1) Baca masukan
masuk = PATH["hasil"] / "tahap01_mentah.parquet"
df = pd.read_parquet(masuk)
u, r = df["ulasan"].astype(str), df["rating"]
n_kata, teks = u.str.split().str.len(), u.str.lower().str.strip()    # kata = dipisah spasi

# 2) Pemeriksaan (data hanya dibaca); label sementara = rating >= tau, hanya untuk audit
dibalas, huruf = df["balasan_dev"].notna(), u.str.lower().map(lambda s: bool(wt.HURUF.search(s)))
n_lbl = (r >= CFG["tau"]).astype(int).groupby(teks).nunique()
med = [float(n_kata[r == k].median()) for k in range(1, 6)]
emoji = int(u.map(lambda s: bool(wt.EMO.search(s) or wt.SIMBOL.search(s))).sum())
AUDIT = [  # (temuan, nilai aktual, kunci TARGET, toleransi, tahap pemakai)
    ("Nilai kosong per kolom", {k: int(v) for k, v in df.isna().sum().items() if v}, "kosong", 0, "2"),
    ("Dibalas pengembang (%), rating 1–4 · 5",
     {"1-4": dibalas[r <= 4].mean() * 100, "5": dibalas[r == 5].mean() * 100}, "dibalas_pct", 0.05, "2"),
    ("Likes bernilai nol (%)", (df["likes"] == 0).mean() * 100, "likes_nol_pct", 0.05, "2"),
    ("Spearman likes–rating", spearmanr(df["likes"], r).statistic, "spearman", 5e-5, "2"),
    ("Nama unik", int(df["nama"].nunique()), "nama_unik", 0, "2"),
    ("Rating 3 >= 8 kata (jumlah · total)", (int(((r == 3) & (n_kata >= 8)).sum()), int((r == 3).sum())),
     "r3_panjang", 0, "3"),
    ("Median kata per rating 1–5", [int(m) if m == int(m) else m for m in med], "median_kata", 0, "6.1"),
    ("Baris berisi emoji atau simbol", emoji, "emoji_simbol", 0, "4"),
    ("Huruf berulang >= 3 (baris · %)", (int(huruf.sum()), huruf.mean() * 100), "huruf_ulang", 0.05, "4"),
    ("Tanda baca berulang (baris)", int(u.map(lambda s: bool(wt.BACA.search(s))).sum()), "baca_ulang", 0, "4"),
    ("Duplikat persis (% baris)", df["ulasan"].duplicated().mean() * 100, "duplikat_pct", 0.05, "5"),
    ("Teks berlabel ganda (teks · baris)", (int((n_lbl == 2).sum()), int(teks.isin(n_lbl[n_lbl == 2].index).sum())),
     "label_ganda", 0, "5"),
]

# 3) Simpan tabel (angka mentah, JSON), tampilkan temuan -> angka -> tahap pemakai, lalu cek
keluar = simpan_tabel(pd.DataFrame([{"temuan": t, "nilai": json.dumps(v), "target": json.dumps(T[k]), "tahap": th}
                                    for t, v, k, _, th in AUDIT]), "tahap01_audit")
judul("1.2", "Audit kualitas data", masukan=[f"{rel(masuk)} ({fmt_id(len(df))} baris)", "lib/wondr_teks.py"],
      keluaran=f"{rel(keluar)} ({len(AUDIT)} baris)")
print(f"{'Temuan':<40}{'Angka':<28}Tahap")
for t, v, _, tol, th in AUDIT:
    print(f"{t:<40}{_tampil(v, 1 if tol == 0.05 else 4):<28}{th}")
cek("jumlah baris tetap", len(df), TARGET["1.1"]["baris"])
for t, v, k, tol, _ in AUDIT:
    cek(t, v, T[k], toleransi=tol)
selesai()

## Tahap 2 — Seleksi atribut

**Dasar naskah:** Bab III tahap 2 (nomor urut sebagai identitas; atribut yang digunakan hanya teks ulasan dan *rating*; nama, balasan pengembang, dan *likes* dihapus; tanggal dan versi hanya informasi pendukung); Bab II subbab 2.4.2. `review_id` dibuat di sini (`CLAUDE.md` Bagian 10, keputusan no. 7). Kode tidak berubah dari v3; `TARGET["2"]` mengikuti data baru (`REVISI_30OKT.md` ✅🔢).

**Package:** `pandas` (olah tabel), `pyarrow` (mesin baca/simpan .parquet); `lib.bantu`.

In [ ]:
# ══ Tahap 2 — Seleksi atribut ══
# Package: pandas (olah tabel), pyarrow (mesin baca/simpan .parquet)
import pandas as pd
mulai()
T = TARGET["2"]

# 1) Baca masukan
masuk = PATH["hasil"] / "tahap01_mentah.parquet"
mentah = pd.read_parquet(masuk)

# 2) Nomor urut sebagai identitas, lalu pilih atribut (ringkasan kalimat Bab III tahap 2)
ATRIBUT = [  # (atribut, peran, alasan)
    ("review_id", "dibuat", "nomor urut sesuai urutan baris data mentah, sebagai identitas"),
    ("ulasan", "dipakai", "teks ulasan: satu-satunya masukan model"),
    ("rating", "dipakai", "sumber label awal (Tahap 3), bukan fitur"),
    ("tanggal", "disimpan", "informasi pendukung (subbab 2.4.2)"),
    ("versi", "disimpan", "informasi pendukung (subbab 2.4.2)"),
    ("nama", "dibuang", "nama pengguna; dihapus untuk menjaga privasi"),
    ("balasan_dev", "dibuang", "ditulis setelah ulasan; isinya dapat membocorkan label"),
    ("likes", "dibuang", "tidak tersedia pada saat prediksi"),
]
pakai = [a for a, peran, _ in ATRIBUT if peran != "dibuang"]
df = mentah.assign(review_id=range(1, len(mentah) + 1))[pakai]

# 3) Simpan
keluar = PATH["hasil"] / "tahap02_atribut.parquet"
df.to_parquet(keluar, index=False)

# 4) Ringkasan + cek
judul("2", "Seleksi atribut", masukan=f"{rel(masuk)} ({fmt_id(len(mentah))} baris, {mentah.shape[1]} kolom)",
      keluaran=f"{rel(keluar)} ({fmt_id(len(df))} baris, {df.shape[1]} kolom)")
print(f"{'Atribut':<13}{'Peran':<10}Alasan")
for a, peran, alasan in ATRIBUT:
    print(f"{a:<13}{peran:<10}{alasan}")
tercatat = {a for a, _, _ in ATRIBUT} - {"review_id"} == set(mentah.columns)
ya = lambda b: "ya" if b else "tidak"
cek("jumlah baris", len(df), T["baris"])
cek("kolom dan urutannya", "sesuai" if list(df.columns) == T["kolom"] else ", ".join(df.columns), "sesuai")
cek("semua kolom data mentah tercatat di tabel", ya(tercatat), "ya")
cek("review_id = 1, 2, ..., n (unik, urut)", ya(df["review_id"].tolist() == list(range(1, len(df) + 1))), "ya")
cek("isi ulasan, rating, tanggal, versi tidak berubah", ya(df[pakai[1:]].equals(mentah[pakai[1:]])), "ya")
selesai()

## Tahap 3 — Pelabelan awal (*top-2-box*)

**Dasar naskah:** Bab III tahap 3; pers. (2.1)–(2.2) dengan τ = 4; Tabel 2.8; Morgan dan Rego (2006). Kolom bernama `label` agar kode acuan `lib/wondr_teks.py` tidak diubah; namanya menjadi `label_awal` di Tahap 9.3. Kode tidak berubah dari v3; `TARGET["3"]` mengikuti data baru (`REVISI_30OKT.md` ✅🔢).

**Package:** `pandas` (olah tabel), `matplotlib` (grafik), `pyarrow` (mesin baca/simpan .parquet); `lib.bantu`. Warna kelas: slot 1–2 palet kategori yang lolos validasi buta warna (Positif biru `#2a78d6`, Non-Positif oranye `#eb6834`), dipakai tetap di semua gambar.

In [ ]:
# ══ Tahap 3 — Pelabelan awal (top-2-box) ══
# Package: pandas (olah tabel), matplotlib (grafik), pyarrow (mesin baca/simpan .parquet)
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
mulai()
T = TARGET["3"]
WARNA = {1: "#2a78d6", 0: "#eb6834"}          # slot kategori 1-2 palet tervalidasi (CVD aman); tetap per kelas

# 1) Baca masukan
masuk = PATH["hasil"] / "tahap02_atribut.parquet"
df = pd.read_parquet(masuk)

# 2) Label awal pers. (2.1)-(2.2): rating >= tau -> 1 (Positif), selain itu 0 (Non-Positif); tidak ada baris dihapus
df["label"] = (df["rating"] >= CFG["tau"]).astype(int)
keluar = PATH["hasil"] / "tahap03_label.parquet"
df.to_parquet(keluar, index=False)

# 3) Gambar: jumlah ulasan per rating, diwarnai per kelas
per = df.groupby("rating").agg(n=("review_id", "size"), kelas=("label", "first"))
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.bar(per.index, per.n, width=0.6, color=[WARNA[k] for k in per.kelas], zorder=2)
for r, n in per.n.items():
    ax.text(r, n + 60, fmt_id(int(n)), ha="center", va="bottom", fontsize=9, color="#52514e")
ax.set(xlabel="Rating", ylabel="Jumlah ulasan", xticks=per.index, ylim=(0, per.n.max() * 1.12))
ax.yaxis.set_major_formatter(lambda y, _: fmt_id(int(y)))
ax.grid(axis="y", color="#e4e3df", linewidth=0.8, zorder=0)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(colors="#52514e", length=0)
ax.legend(handles=[Patch(color=WARNA[k], label=CFG["kelas"][k]) for k in (1, 0)], frameon=False, loc="upper left")
gambar = simpan_gambar(fig, "tahap03_distribusi_rating")

# 4) Ringkasan + cek
kelas = {CFG["kelas"][k]: (int((df.label == k).sum()), (df.label == k).mean() * 100) for k in (1, 0)}
judul("3", "Pelabelan awal (top-2-box)", masukan=f"{rel(masuk)} ({fmt_id(len(df))} baris)",
      keluaran=[f"{rel(keluar)} ({fmt_id(len(df))} baris, kolom baru: label)", rel(gambar)])
print(f"{'Rating':<8}{'Jumlah':>8}  Kelas")
for r, (n, k) in per.iterrows():
    print(f"{r:<8}{fmt_id(int(n)):>8}  {CFG['kelas'][k]}")
for nama, (n, pct) in kelas.items():
    print(f"Total {nama:<12}{fmt_id(n):>7}  ({fmt_id(pct, 2)}%)")
cek("jumlah baris tetap (tidak ada yang dihapus)", len(df), TARGET["2"]["baris"])
cek("jumlah per rating", {int(r): int(n) for r, n in per.n.items()}, T["per_rating"])
cek("Positif (jumlah · %)", kelas["Positif"], T["positif"], toleransi=0.005)
cek("Non-Positif (jumlah · %)", kelas["Non-Positif"], T["non_positif"], toleransi=0.005)
cek("gambar tersimpan", "ya" if gambar.exists() else "tidak", "ya")
selesai()

## Tahap 4 — *Pre-processing*

**Dasar naskah:** Bab III tahap 4; Bab II subbab 2.2 dan 2.2.4 (Tabel 2.1–2.7); Brody dan Diakopoulos (2011). Enam langkah `wt.langkah(KAMUS)` dijalankan berurutan, lalu teks kosong dibuang. Langkah 4 = normalisasi pemanjangan kata: kamus Brody lebih dulu, lalu cadangan ≥ 3 → 2 untuk kata memanjang yang tidak ada di kamus (`REVISI_BRODY.md` R13–R15). Kamus dibentuk sekali dari teks langkah 1–3 seluruh 9.672 ulasan (hanya frekuensi kata; label dan *rating* tidak dipakai), disimpan, lalu dibekukan (R14). Tanpa *stemming*, tanpa *stopword removal*, dan tanpa normalisasi kamus slang.

**Package:** `importlib`, `collections` (bawaan Python); `pandas`; `lib.wondr_teks` (memakai `re`, `unicodedata`); `lib.bantu`. Masukan uji contoh Bab II ditulis sebagai *escape* Unicode.

Sel 1/2 membentuk kamus dan menyimpannya ke `hasil/tahap04_kamus_brody.json` dan `bab4/tabel/tahap04_kamus_brody.csv`. Sel 2/2 membaca kamus beku itu, menjalankan enam langkah, dan menyimpan `hasil/tahap04_bersih.parquet`.

In [ ]:
# ══ Tahap 4 — Pre-processing (sel 1/2: kamus pemanjangan kata Brody) ══
# Package: importlib, collections (bawaan Python); pandas; lib.wondr_teks (sebelum_kamus, kamus_brody, ringkas, KATA)
import importlib
from collections import Counter, defaultdict
import pandas as pd
import lib.wondr_teks as wt
importlib.reload(wt)                                        # pakai isi lib/wondr_teks.py yang terbaru
mulai()
T, ya = TARGET["4"], (lambda x: "ya" if x else "tidak")
M = lambda w: bool(wt.HURUF.search(w))                      # kata memanjang: ada huruf berulang >= 3

# 1) Kamus dari teks langkah 1-3 SELURUH ulasan (R14): hanya frekuensi kata; label dan rating tidak dibaca
masuk = PATH["hasil"] / "tahap03_label.parquet"
pra = pd.read_parquet(masuk, columns=["ulasan"])["ulasan"].astype(str).map(wt.sebelum_kamus)
KAMUS, info = wt.kamus_brody(pra, CFG["brody_ambang"])
K2, i2 = wt.kamus_brody(pra, CFG["brody_ambang"])          # dibentuk ulang: harus identik

# 2) Rincian per kelompok (untuk tabel dan cek): frekuensi semua bentuk, kata memanjang di teks dan jalurnya
frek = Counter(w for t in pra for w in wt.KATA.findall(t))
grup = defaultdict(dict)
for w, n in frek.items():
    grup[wt.ringkas(w)][w] = n
mem = [w for t in pra for w in wt.KATA.findall(t) if M(w)]
lk, cad = [w for w in mem if wt.ringkas(w) in KAMUS], [w for w in mem if wt.ringkas(w) not in KAMUS]
nb = [len(set(x)) for x in (mem, lk, cad)]                  # bentuk unik: total, lewat kamus, lewat cadangan
def baris(k, v):                                            # satu baris tabel kamus
    pm = sorted(((w, n) for w, n in grup[k].items() if M(w)), key=lambda x: (-x[1], x[0]))
    return {"kanonik": v, "frekuensi_kanonik": frek[v], "n_bentuk_memanjang": len(pm),
            "kemunculan_memanjang": sum(n for _, n in pm), "contoh": ", ".join(w for w, _ in pm[:3])}
tab = (pd.DataFrame([baris(k, v) for k, v in KAMUS.items()])
         .sort_values(["kemunculan_memanjang", "kanonik"], ascending=[False, True], ignore_index=True))
seri = sum(sorted(grup[k].values(), reverse=True)[:2].count(frek[v]) > 1 for k, v in KAMUS.items())

# 3) Simpan kamus beku (dipakai sel 2/2 dan ulasan baru) dan tabelnya
simpan_json({"ambang": CFG["brody_ambang"], "info": info, "kamus": KAMUS}, PATH["hasil"] / "tahap04_kamus_brody.json")
keluar = simpan_tabel(tab, "tahap04_kamus_brody")

# 4) Ringkasan + cek
judul("4", "Pre-processing (sel 1/2: kamus Brody)", masukan=f"{rel(masuk)} ({fmt_id(len(pra))} ulasan; langkah 1–3)",
      keluaran=[f"hasil/tahap04_kamus_brody.json ({fmt_id(len(KAMUS))} entri)", rel(keluar)])
print(f"Kamus (ambang frekuensi {CFG['brody_ambang']}): kelompok memanjang {info['kelompok_memanjang']} · "
      f"dipakai {len(KAMUS)} · dibuang: frekuensi < {CFG['brody_ambang']} {info['buang_frekuensi']}, "
      f"kanonik 1 huruf {info['buang_1_huruf']}, kanonik memanjang {info['buang_kanonik_memanjang']}")
print(f"Kata memanjang: {fmt_id(len(mem))} kemunculan ({nb[0]} bentuk) = lewat kamus {len(lk)} ({nb[1]}) "
      f"+ cadangan >= 3 -> 2 {len(cad)} ({nb[2]})")
print(f"\n{'kanonik':<12}{'frekuensi':>10}{'bentuk':>8}{'kemunculan':>12}  contoh bentuk memanjang")
for r in tab.head(12).itertuples():
    print(f"{r.kanonik:<12}{fmt_id(r.frekuensi_kanonik):>10}{r.n_bentuk_memanjang:>8}{r.kemunculan_memanjang:>12}  {r.contoh}")
cek("isi kamus (info · dipakai)", {**info, "dipakai": len(KAMUS)}, T["kamus"])
cek("kata memanjang: total · kamus · cadangan · bentuk", [len(mem), len(lk), len(cad), tuple(nb)],
    [T["memanjang"][k] for k in ("total", "kamus", "cadangan", "bentuk")])
cek("lewat kamus + cadangan = total (kemunculan · bentuk)", [ya(len(lk) + len(cad) == len(mem)), ya(nb[1] + nb[2] == nb[0])],
    ["ya", "ya"])
V = list(KAMUS.values())
cek("kanonik: > 1 huruf · tanpa huruf berulang >= 3 · frekuensi >= ambang · seri", [ya(all(len(v) > 1 for v in V)),
    ya(not any(map(M, V))), ya(all(frek[v] >= CFG["brody_ambang"] for v in V)), seri], ["ya", "ya", "ya", 0])
cek("kamus dibentuk ulang identik (deterministik) · JSON = kamus", [ya(K2 == KAMUS and i2 == info),
    ya(baca_json(PATH["hasil"] / "tahap04_kamus_brody.json")["kamus"] == KAMUS)], ["ya", "ya"])
selesai()

In [ ]:
# ══ Tahap 4 — Pre-processing (sel 2/2: enam langkah dengan kamus beku) ══
# Package: importlib (bawaan Python); pandas; lib.wondr_teks (langkah, bersihkan, ringkas; memakai re, unicodedata)
import importlib
import pandas as pd
import lib.wondr_teks as wt
importlib.reload(wt)                                        # pakai isi lib/wondr_teks.py yang terbaru
mulai()
T = TARGET["4"]

# 1) Kamus beku dari sel 1/2; uji bersihkan() dengan contoh Bab II (Tabel 2.1-2.7), masukan ditulis escape Unicode
KB = baca_json(PATH["hasil"] / "tahap04_kamus_brody.json")
KAMUS = KB["kamus"]
CONTOH = ["\U0001D401\U0001D400\U0001D406\U0001D414\U0001D412\U0001D412\U0001D412 bgt!!! \u2605\u2605 tp "
          "kadang\u00b2 error, nunggu lamaaa", "\uff4d\uff41\uff4e\uff54\uff41\uff50"]
uji = [wt.bersihkan(x, KAMUS) for x in CONTOH]

# 2) Baca masukan, jalankan 6 langkah berurutan sambil menghitung ulasan yang berubah
masuk = PATH["hasil"] / "tahap03_label.parquet"
df = pd.read_parquet(masuk)
s, berubah = df["ulasan"].astype(str), []
for i, (_, f) in enumerate(wt.langkah(KAMUS)):
    t = s.map(f)
    berubah.append(int((t != s).sum()))
    if i == 3:                                              # langkah 4: simpan contoh sebelum -> sesudah
        contoh4 = pd.DataFrame({"id": df["review_id"], "a": s, "b": t})[t != s].head(3)
    s = t
df["text_clean"] = s

# 3) Buang teks kosong (jumlah sebelum dan sesudah ditampilkan), lalu simpan
kosong, bersih = df[df["text_clean"] == ""], df[df["text_clean"] != ""].reset_index(drop=True)
keluar = PATH["hasil"] / "tahap04_bersih.parquet"
bersih.to_parquet(keluar, index=False)

# 4) Ringkasan + cek
judul("4", "Pre-processing (sel 2/2: enam langkah)", masukan=[f"{rel(masuk)} ({fmt_id(len(df))} baris)",
      f"hasil/tahap04_kamus_brody.json ({fmt_id(len(KAMUS))} entri)"],
      keluaran=f"{rel(keluar)} ({fmt_id(len(bersih))} baris, kolom baru: text_clean)")
print(f"{'Langkah':<40}{'Ulasan berubah':>15}")
for (nama, _), n in zip(wt.langkah(KAMUS), berubah):
    print(f"{nama:<40}{fmt_id(n):>15}")
r_kosong = {int(k): int(v) for k, v in kosong["rating"].value_counts().items()}
print(f"\nTeks kosong dibuang: {fmt_id(len(kosong))} ({', '.join(f'rating {k}: {v}' for k, v in r_kosong.items())})"
      f"  →  {fmt_id(len(df))} menjadi {fmt_id(len(bersih))} ulasan")
print("Contoh langkah 4 saja (kamus Brody, sisanya >= 3 -> 2); tanda baca berulang baru dipadatkan di langkah 5:")
lewat = lambda x: ("kamus" if any(wt.ringkas(w) in KAMUS for w in wt.KATA.findall(x) if wt.HURUF.search(w))
                   else "cadangan")                         # jalur kata memanjang di langkah 4
for i, a, b in contoh4.itertuples(index=False):                # teks utuh, tanpa pemotongan
    kata = "; ".join(f'"{x}" → "{y}" ({lewat(x)})' for x, y in zip(a.split(), b.split()) if x != y)
    print(f"  [{i}] kata yang berubah : {kata}")
    print(f"       hasil 6 langkah  : {bersih.loc[bersih['review_id'] == i, 'text_clean'].iloc[0]}")
cek("kamus beku: ambang · entri", [KB["ambang"], len(KAMUS)], [CFG["brody_ambang"], T["kamus"]["dipakai"]])
cek("uji contoh Bab II (Tabel 2.1-2.7)", uji, T["contoh_bab2"])
cek("ulasan berubah per langkah", berubah, T["berubah"])
cek("teks kosong dibuang", len(kosong), T["kosong"])
cek("rating teks kosong", r_kosong, T["kosong_rating"])
cek("jumlah setelah pre-processing", len(bersih), T["hasil"])
tc = bersih["text_clean"]                                      # invarian: aturan berlaku di seluruh teks
cek("sisa huruf berulang · tanda baca berulang · emoji", [int(tc.map(lambda x: bool(p.search(x))).sum())
    for p in (wt.HURUF, wt.BACA, wt.EMO)], [0, 0, 0])
sama = (bersih["ulasan"].map(lambda x: wt.bersihkan(x, KAMUS)) == tc).all()
cek("text_clean = bersihkan(ulasan, KAMUS) di semua baris", "ya" if sama else "tidak", "ya")
selesai()

## Tahap 5 — Deduplikasi dan cek kunci yang menggabungkan bentuk berbeda

**Dasar naskah:** Bab III tahap 5; Bab II subbab 2.4.1 dan Tabel 2.11 (label mayoritas; kunci seri dikeluarkan; wakil dipilih berdasarkan kesesuaian dengan label mayoritas, teks tersering, lalu nomor urut terkecil). Tabel kunci seri ditampilkan sebagai *program saja* (`CLAUDE.md` Bagian 10, keputusan no. 10). Kode deduplikasi tidak berubah dari v3; `TARGET["5"]` mengikuti data baru (`REVISI_30OKT.md` ✅🔢; `REVISI_BRODY.md` Bagian 5).

Sel 2/2 (baru, `REVISI_30OKT.md` R9 dan Bagian 5.1; *program saja*, data tidak diubah): kunci deduplikasi yang menggabungkan lebih dari satu bentuk teks (`k_asli` = a-z, 0-9, spasi tanpa peringkasan karakter) ditampilkan seluruhnya dan disimpan ke `bab4/tabel/tahap05_kunci_gabungan.csv`. Keputusan penggabungan tetap di tangan peneliti; tidak ada yang diubah otomatis.

**Package:** `importlib`, `re` (bawaan Python); `pandas`; `lib.wondr_teks` (`kunci_dedup`, `deduplikasi`); `lib.bantu`.

In [ ]:
# ══ Tahap 5 — Deduplikasi ══
# Package: importlib (bawaan Python); pandas; lib.wondr_teks (kunci_dedup, deduplikasi)
import importlib
import pandas as pd
import lib.wondr_teks as wt
importlib.reload(wt)                                        # pakai isi lib/wondr_teks.py yang terbaru
mulai()
T = TARGET["5"]

# 1) Baca masukan
masuk = PATH["hasil"] / "tahap04_bersih.parquet"
d = pd.read_parquet(masuk)

# 2) Deduplikasi (kode acuan): label mayoritas, kunci seri dikeluarkan, wakil = teks tersering lalu id terkecil
hasil, suara, seri = wt.deduplikasi(d)
keluar = PATH["hasil"] / "tahap05_dedup.parquet"
hasil.to_parquet(keluar, index=False)

# 3) Rincian alur, dihitung dari tabel suara per kunci; simpan CSV (angka mentah)
kunci, sah = d["text_clean"].map(wt.kunci_dedup), suara[~seri]
ALUR = {"ulasan masuk": len(d), "kunci kosong (dibuang)": int((kunci == "").sum()),
        "kunci deduplikasi": len(suara), "kunci seri": int(seri.sum()),
        "dibuang: salinan berlabel sama": int((sah[["n_np", "n_p"]].max(axis=1) - 1).sum()),
        "dibuang: anggota kalah suara": int(sah[["n_np", "n_p"]].min(axis=1).sum()),
        "dibuang: anggota kunci seri": int(suara.loc[seri, ["n_np", "n_p"]].to_numpy().sum()),
        "dokumen hasil": len(hasil)}
alur = simpan_tabel(pd.DataFrame({"langkah": list(ALUR), "jumlah": list(ALUR.values())}), "tahap05_alur")

# 4) Ringkasan + cek
dibuang = [len(d) - ALUR["kunci kosong (dibuang)"] - len(hasil)] + [v for k, v in ALUR.items() if k.startswith("dibuang")]
pos = (d["label"].mean() * 100, hasil["label"].mean() * 100)
judul("5", "Deduplikasi", masukan=f"{rel(masuk)} ({fmt_id(len(d))} baris)",
      keluaran=[f"{rel(keluar)} ({fmt_id(len(hasil))} baris, kolom baru: dedup_key)", rel(alur)])
for k, v in ALUR.items():
    print(f"{k:<34}{fmt_id(v):>7}")
print(f"Proporsi Positif                  {fmt_id(pos[0], 1)}% → {fmt_id(pos[1], 1)}%")
print("\nKunci seri yang dikeluarkan (program saja; rating anggotanya saling bertentangan):")
anggota = d.assign(k=kunci)[kunci.isin(suara[seri].index)].sort_values("review_id")
for k, g in anggota.groupby("k"):
    print(f"  {k:<20}" + " · ".join(f"[{i}] rating {r}" for i, r in zip(g["review_id"], g["rating"])))
cek("kunci kosong dibuang", ALUR["kunci kosong (dibuang)"], 0)
cek("kunci · kunci seri · ulasan seri", (len(suara), int(seri.sum()), dibuang[3]),
    (T["kunci"], T["kunci_seri"], T["ulasan_seri"]))
cek("jumlah dokumen", len(hasil), T["dokumen"])
cek("dibuang (total · salinan · kalah suara · seri)", dibuang, T["dibuang"])
cek("total dibuang = jumlah ketiga rincian", "ya" if dibuang[0] == sum(dibuang[1:]) else "tidak", "ya")
cek("Positif (%) sebelum · sesudah", pos, T["positif_pct"], toleransi=0.05)
cek("satu dokumen per kunci", "ya" if hasil["dedup_key"].is_unique else "tidak", "ya")
selesai()

In [ ]:
# ══ Tahap 5 — Cek kunci yang menggabungkan bentuk berbeda (tambahan; program saja, data tidak diubah) ══
# Package: importlib, re (bawaan Python); pandas; lib.wondr_teks (kunci_dedup)
import importlib, re
import pandas as pd
import lib.wondr_teks as wt
importlib.reload(wt)                                        # pakai isi lib/wondr_teks.py yang terbaru
mulai()
T, ya = TARGET["5"], (lambda x: "ya" if x else "tidak")

# 1) Dua kunci per ulasan: k_asli = a-z, 0-9, spasi TANPA peringkasan karakter; k_dedup = kunci_dedup() (dengan peringkasan)
masuk = PATH["hasil"] / "tahap04_bersih.parquet"
d = pd.read_parquet(masuk, columns=["review_id", "text_clean"])
d["k_asli"] = d["text_clean"].map(lambda s: re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", s)).strip())
d["k_dedup"] = d["text_clean"].map(wt.kunci_dedup)
d = d[d["k_dedup"] != ""]

# 2) Kunci terdampak = k_dedup dengan > 1 k_asli berbeda; bentuk diurutkan menurut jumlah ulasan (menurun), lalu abjad
bentuk = d.groupby(["k_dedup", "k_asli"]).size().rename("n").reset_index()
n_bentuk = bentuk.groupby("k_dedup").size()
gab = (bentuk[bentuk["k_dedup"].isin(n_bentuk[n_bentuk > 1].index)]
       .sort_values(["k_dedup", "n", "k_asli"], ascending=[True, False, True]))
gab["teks"] = gab["k_asli"] + " (" + gab["n"].astype(str) + ")"
tab = (gab.groupby("k_dedup", sort=True).agg(n_bentuk=("k_asli", "size"), n_ulasan=("n", "sum"), bentuk=("teks", " | ".join))
          .reset_index().rename(columns={"k_dedup": "kunci"}))
tab["memuat_angka"] = tab["kunci"].str.contains(r"\d")
keluar = simpan_tabel(tab, "tahap05_kunci_gabungan")

# 3) Ringkasan + cek (keputusan penggabungan tetap di tangan peneliti; tidak ada yang diubah otomatis)
judul("5", "Cek kunci yang menggabungkan bentuk berbeda", masukan=f"{rel(masuk)} ({fmt_id(len(d))} ulasan berkunci)",
      keluaran=f"{rel(keluar)} ({len(tab)} kunci)")
print(f"Kunci terdampak: {len(tab)} dari {fmt_id(d['k_dedup'].nunique())} kunci · "
      f"memuat angka: {int(tab['memuat_angka'].sum())} · ulasan di kunci terdampak: {fmt_id(int(tab['n_ulasan'].sum()))}")
print(f"\n{'kunci':<26}bentuk (jumlah ulasan)")
for r in tab.itertuples():
    print(f"{r.kunci:<26}{r.bentuk}")
cek("kunci terdampak · memuat angka", [len(tab), int(tab["memuat_angka"].sum())], list(T["gabungan"]))
cek("jumlah kunci = Tahap 5 · kunci_dedup(k_asli) = k_dedup", [d["k_dedup"].nunique(),
    ya((d["k_asli"].map(wt.kunci_dedup) == d["k_dedup"]).all())], [T["kunci"], "ya"])
selesai()

## Tahap 6.1 — Pembagian data

**Dasar naskah:** Bab III tahap 6; pers. (2.6); Tabel 2.12–2.13. Latih 70%, validasi 15%, uji 15%; grup berkunci sama ditempatkan pada himpunan yang sama; stratifikasi pada tingkat grup berdasarkan label mayoritas grup dan kelompok panjang (1–2, 3–10, >10 kata); grup setiap strata diacak dengan *seed* 42 lalu dialokasikan ke himpunan dengan tingkat pengisian terendah. Kode tidak berubah dari v3; `TARGET["6.1"]` mengikuti data baru (`REVISI_30OKT.md` ✅🔢; `REVISI_BRODY.md` Bagian 5).

**Package:** `importlib` (bawaan Python); `pandas`; `lib.wondr_teks` (`bagi_data`, `kunci_grup`, `kelompok_panjang`; memakai `numpy`); `lib.bantu`.

In [ ]:
# ══ Tahap 6.1 — Pembagian data ══
# Package: importlib (bawaan Python); pandas; lib.wondr_teks (bagi_data, kunci_grup, kelompok_panjang)
import importlib
import pandas as pd
import lib.wondr_teks as wt
importlib.reload(wt)                                        # pakai isi lib/wondr_teks.py yang terbaru
mulai()
T = TARGET["6.1"]
KOLOM = ["review_id", "ulasan", "text_clean", "rating", "label", "group_key", "dedup_key", "kelompok_panjang"]

# 1) Baca masukan
masuk = PATH["hasil"] / "tahap05_dedup.parquet"
dd = pd.read_parquet(masuk)

# 2) Bagi berbasis grup pers. (2.6), stratifikasi label grup x kelompok panjang (kode acuan; seed dari CFG)
bagi, grup = wt.bagi_data(dd, seed=CFG["seed_bagi"])

# 3) Simpan tiga bagian
keluar = {b: PATH["hasil"] / f"tahap06_{b}.parquet" for b in CFG["proporsi"]}
for b, p in keluar.items():
    bagi.loc[bagi["split"] == b, KOLOM].reset_index(drop=True).to_parquet(p, index=False)

# 4) Ringkasan + cek
hit = {b: (int((bagi.split == b).sum()), int(((bagi.split == b) & (bagi.label == 0)).sum()),
           int(((bagi.split == b) & (bagi.label == 1)).sum())) for b in CFG["proporsi"]}
judul("6.1", "Pembagian data", masukan=f"{rel(masuk)} ({fmt_id(len(dd))} dokumen)",
      keluaran=[f"{rel(p)} ({fmt_id(hit[b][0])} dokumen)" for b, p in keluar.items()])
print(f"Grup: {fmt_id(len(grup))} dari {fmt_id(len(dd))} dokumen "
      f"({fmt_id(int((grup.n > 1).sum()))} grup berisi lebih dari satu dokumen)\n")
print(f"{'Bagian':<10}{'Dokumen':>8}{'Porsi':>8}{'Target':>8}{'Non-Positif':>13}{'Positif':>9}")
for b, p in CFG["proporsi"].items():
    n, n0, n1 = hit[b]
    print(f"{b:<10}{fmt_id(n):>8}{fmt_id(n / len(bagi) * 100, 1) + '%':>8}{fmt_id(p * 100, 0) + '%':>8}"
          f"{fmt_id(n0):>13}{fmt_id(n1):>9}")
cek("PROPORSI di lib/wondr_teks.py = CFG", wt.PROPORSI, CFG["proporsi"])
cek("jumlah grup", len(grup), T["grup"])
for b in CFG["proporsi"]:
    cek(f"{b} (dokumen · Non-Positif · Positif)", hit[b], T[b])
cek("semua dokumen mendapat satu bagian", "ya" if bagi["split"].notna().all() and len(bagi) == len(dd) else "tidak", "ya")
selesai()

## Tahap 6.2 — Cek kebocoran dan keseimbangan

**Dasar naskah:** Bab III tahap 6 (setelah pembagian dipastikan tidak ada kunci pengelompokan, teks, dan kunci deduplikasi yang muncul pada lebih dari satu himpunan); Bab II subbab 2.4.2–2.4.4. Kedua sel hanya **membaca** berkas Tahap 6.1; *seed*, proporsi, dan algoritma pembagian tidak diubah. Cek kembaran-dekat dihapus (`REVISI_30OKT.md` R10); `TARGET["6.2"]` mengikuti data baru (`REVISI_BRODY.md` Bagian 5).

**Package:** `pandas` (olah tabel); `scipy.stats.chi2_contingency` (uji khi-kuadrat); `lib.bantu`.

Sel 1/2: kebocoran `group_key`, `dedup_key`, `text_clean` dan kelengkapan dokumen. Sel 2/2: proporsi label dan kelompok panjang per bagian, uji khi-kuadrat, dan tabel strata. **Mulai di sini data uji dikunci.**

In [ ]:
# ══ Tahap 6.2 — Cek kebocoran dan keseimbangan (sel 1/2: kebocoran dan kelengkapan) ══
# Package: pandas (olah tabel, baca .parquet)
import pandas as pd
mulai()
T, BAGIAN = TARGET["6.2"], list(CFG["proporsi"])

# 1) Baca ketiga bagian dari berkas Tahap 6.1 (pembagian tidak diubah) dan hasil deduplikasi
masuk = {b: PATH["hasil"] / f"tahap06_{b}.parquet" for b in BAGIAN}
df = pd.concat([pd.read_parquet(p).assign(bagian=b) for b, p in masuk.items()], ignore_index=True)
dedup = pd.read_parquet(PATH["hasil"] / "tahap05_dedup.parquet")

# 2) Kebocoran: kunci yang muncul di lebih dari satu bagian
KUNCI = ("group_key", "dedup_key", "text_clean")
lintas = {k: df.groupby(k)["bagian"].nunique() for k in KUNCI}
bocor = {k: s[s > 1] for k, s in lintas.items()}

# 3) Ringkasan + cek
judul("6.2", "Cek kebocoran (sel 1/2)", masukan=[f"{rel(p)} ({fmt_id(int((df.bagian == b).sum()))} dokumen)"
                                                for b, p in masuk.items()])
print(f"{'Kunci':<12}{'Nilai unik':>11}{'Di > 1 bagian':>15}")
for k in KUNCI:
    print(f"{k:<12}{fmt_id(len(lintas[k])):>11}{fmt_id(len(bocor[k])):>15}")
for k in KUNCI:                                             # contoh hanya dicetak bila ada kebocoran
    for kunci in bocor[k].index[:5]:
        print(f"  BOCOR {k} {kunci!r}: {sorted(df.loc[df[k] == kunci, 'bagian'].unique())}")
grup_per_bagian = int(df.groupby("bagian")["group_key"].nunique().sum())
for k in KUNCI:
    cek(f"{k} di lebih dari satu bagian", len(bocor[k]), T["bocor"])
cek("jumlah grup per bagian dijumlahkan", grup_per_bagian, TARGET["6.1"]["grup"])
cek("total dokumen · review_id unik", (len(df), int(df["review_id"].nunique())),
    (TARGET["5"]["dokumen"], TARGET["5"]["dokumen"]))
sama = set(df["review_id"]) == set(dedup["review_id"])
cek("review_id = review_id hasil deduplikasi", "ya" if sama else "tidak", "ya")
selesai()

In [ ]:
# ══ Tahap 6.2 — Cek kebocoran dan keseimbangan (sel 2/2: keseimbangan dan strata) ══
# Package: pandas (olah tabel); scipy.stats.chi2_contingency (uji khi-kuadrat)
import pandas as pd
from scipy.stats import chi2_contingency
mulai()
T, BAGIAN, PANJANG = TARGET["6.2"], list(CFG["proporsi"]), list(CFG["kelompok_panjang"])

# 1) Baca ketiga bagian (pembagian tidak diubah)
df = pd.concat([pd.read_parquet(PATH["hasil"] / f"tahap06_{b}.parquet").assign(bagian=b) for b in BAGIAN],
               ignore_index=True)

# 2) Tabel silang bagian x label dan bagian x kelompok panjang, lalu uji khi-kuadrat
x_label = pd.crosstab(df["bagian"], df["label"]).loc[BAGIAN]
x_panjang = pd.crosstab(df["bagian"], df["kelompok_panjang"]).loc[BAGIAN, PANJANG]
p_label, p_panjang = chi2_contingency(x_label)[1], chi2_contingency(x_panjang)[1]
pct_np = x_label[0] / x_label.sum(axis=1) * 100
pct_pj = x_panjang.div(x_panjang.sum(axis=1), axis=0) * 100

# 3) Simpan tabel strata (label x kelompok panjang) per bagian, angka mentah
strata = pd.crosstab([df["label"].map(CFG["kelas"]), df["kelompok_panjang"]], df["bagian"])[BAGIAN]
keluar = simpan_tabel(strata.reset_index().rename(columns={"label": "kelas"}), "tahap06_strata")

# 4) Ringkasan + cek
judul("6.2", "Cek keseimbangan (sel 2/2)", masukan="hasil/tahap06_{latih,validasi,uji}.parquet", keluaran=rel(keluar))
print(f"{'Bagian':<10}{'Non-Positif':>12}" + "".join(f"{'kata ' + k:>12}" for k in PANJANG))
for b in BAGIAN:
    sel = [fmt_id(pct_np[b], 1)] + [fmt_id(pct_pj.loc[b, k], 1) for k in PANJANG]
    print(f"{b:<10}" + "".join(f"{x + '%':>12}" for x in sel))
print(f"\nKhi-kuadrat: bagian x label p = {fmt_id(p_label)} · bagian x kelompok panjang p = {fmt_id(p_panjang)}")
print(f"\nDokumen per strata\n{strata.to_string()}")
cek("Non-Positif (%) latih · validasi · uji", [pct_np[b] for b in BAGIAN], T["non_positif_pct"], toleransi=0.05)
for b in BAGIAN:
    cek(f"kelompok panjang (%) {b}", list(pct_pj.loc[b]), T["panjang_pct"][b], toleransi=0.05)
cek("p khi-kuadrat label · panjang", (p_label, p_panjang), (T["p_label"], T["p_panjang"]), toleransi=5e-5)
print("\nMulai di sini data uji dikunci: label dan metriknya tidak dipakai sampai Tahap 14 "
      "(kecuali teksnya diekspor ke lembar anotasi di Tahap 9.1).")
selesai()